# AlphaZero TicTacToe — Self-Play Training

Trains a `PolicyValueNetwork` (10-block ResNet, 256 filters, ~11.8M params) via AlphaZero-style
self-play on a Colab GPU. Each iteration runs N self-play games, collects
`(board_encoding, MCTS_visit_distribution, game_outcome)` examples, augments them with all
8 board symmetries, then performs mini-batch SGD.

**Checkpoints are saved to Google Drive** — set `RESUME_FROM` in the Hyperparameters cell to
continue training from a saved checkpoint.

## What this optimises for

| Loss term | Formula | Goal |
|---|---|---|
| Policy loss | `CrossEntropy(logits, MCTS_visit_distribution)` | Network move priors match tree search |
| Value loss | `MSE(tanh(value_head), game_outcome ∈ {-1,0,1})` | Accurate position evaluation |

**Training success (3×3):** win rate vs Random ≥ 95%, win rate vs MinimaxAB(depth=4) ≥ 50%,
self-play draw rate ≥ 70% (TicTacToe is a theoretical draw under optimal play).

**Estimated runtime (Colab T4 GPU, 3×3, default settings):** ~1–2 min/iteration → ~3–5 hours for 150 iterations.

In [ ]:
# ── Cell 1: GPU check ──────────────────────────────────────────────────────
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram_gb:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU before proceeding.')

In [ ]:
# ── Cell 2: Mount Google Drive ────────────────────────────────────────────
import os
from google.colab import drive

drive.mount('/content/drive')
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/alphazero_checkpoints'
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint directory: {DRIVE_CHECKPOINT_DIR}')

In [ ]:
# ── Cell 3: Clone repository ──────────────────────────────────────────────
# Replace the URL with your actual repository URL.
# If your repo is private, use:
#   !git clone https://<TOKEN>@github.com/<USER>/TicTacToe.git /content/TicTacToe
# Or upload a zip and extract it:
#   from google.colab import files; files.upload()  # upload TicTacToe.zip
#   !unzip -q TicTacToe.zip -d /content/TicTacToe

import subprocess, sys

REPO_URL = 'https://github.com/YOUR_USERNAME/TicTacToe.git'  # <-- edit this

if not os.path.exists('/content/TicTacToe'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/TicTacToe'], check=True)
else:
    print('Repository already cloned.')

sys.path.insert(0, '/content/TicTacToe')
print('Path set.')

In [ ]:
# ── Cell 4: Hyperparameters ───────────────────────────────────────────────
# This is the ONLY cell you should normally edit between runs.

# Board configuration
N = 3                        # Board dimension (e.g. 3, 5, 7)
K = 3                        # Win length (K <= N; e.g. 3 for standard TicTacToe, 5 for Gomoku)
NETWORK_TYPE = 'float32'     # MUST be 'float32' for training from scratch.
                             # Alternative: 'bitnet' (BitNet Transformer, slower to train).
                             # Do NOT use 'quantized' — its weights are frozen int8 buffers.

# MCTS settings
NUM_SIMULATIONS = 100        # MCTS simulations per move (higher = stronger data, slower)
C_PUCT = 1.0                 # PUCT exploration constant
TEMPERATURE = 1.0            # Move temperature for first 10 moves; anneals to 0.1 afterwards

# Training loop
NUM_ITERATIONS = 150         # Total training iterations
GAMES_PER_ITERATION = 25     # Self-play games per iteration
BATCH_SIZE = 64              # Training mini-batch size
LR = 1e-3                    # Adam learning rate
BUFFER_MAX_SIZE = 50_000     # Max examples in replay buffer (rolling window)

# Evaluation & checkpointing
EVAL_EVERY = 5               # Evaluate agent every N iterations
EVAL_GAMES = 40              # Games per evaluation matchup (20 as X, 20 as O)
CHECKPOINT_EVERY = 10        # Save checkpoint every N iterations

# Resume from checkpoint (set to path string without .pt extension, or leave as None)
# Example: '/content/drive/MyDrive/alphazero_checkpoints/checkpoint_iter_0050'
RESUME_FROM = None

# ── Hyperparameter guide for larger boards ────────────────────────────────
# Board   | NUM_SIMULATIONS | GAMES_PER_ITER | NUM_ITERATIONS | BATCH_SIZE | BUFFER_MAX
# 3x3 k=3 |      100        |       25       |      150       |     64     |   50_000
# 5x5 k=4 |      200        |       50       |      300       |    128     |  100_000
# 7x7 k=5 |      400        |       50       |      500       |    256     |  200_000

print(f'Config: n={N}, k={K}, network={NETWORK_TYPE}')
print(f'Training: {NUM_ITERATIONS} iters x {GAMES_PER_ITERATION} games x {NUM_SIMULATIONS} sims')

In [ ]:
# ── Cell 5: Imports ───────────────────────────────────────────────────────
import json
import math
import random
import time

import matplotlib.pyplot as plt
import torch

from tictactoe.agents._search_budget import SearchBudget
from tictactoe.agents._shared_utils import check_forced_move
from tictactoe.agents.classic_search.minimax_ab import MinimaxAB
from tictactoe.agents.random_agent import RandomAgent
from tictactoe.agents.reinforcement_learning.alphazero import AlphaZeroAgent, PUCTNode
from tictactoe.agents.reinforcement_learning.shared.neural_net import (
    DEVICE, encode_board_flat,
)
from tictactoe.agents.reinforcement_learning.shared.self_play_trainer import SelfPlayTrainer
from tictactoe.core.board import Board
from tictactoe.core.types import Cell, Player, Result

print('All imports OK.')
print(f'Torch version: {torch.__version__}')
print(f'DEVICE (from neural_net): {DEVICE}')

In [ ]:
# ── Cell 6: TrainableAlphaZeroAgent ──────────────────────────────────────
# Subclass of AlphaZeroAgent that records MCTS visit counts during self-play
# so we can construct policy targets for training.
#
# Why we can't just call super().choose_move(): the parent builds a local
# `root` node, runs MCTS, then discards it — self._cached_root stores only
# the SELECTED child, not the root visit distribution we need. The override
# copies the parent body verbatim and inserts buffer recording before return.

class TrainableAlphaZeroAgent(AlphaZeroAgent):
    """AlphaZeroAgent that records (state, policy, board, player) during play.

    After each self-play game, call collect_examples(result) to get fully
    labelled (state_enc, policy_target, target_value) training examples.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._episode_buffer: list = []

    def clear_episode_buffer(self):
        """Reset buffer and MCTS tree cache at the start of each game."""
        self._episode_buffer = []
        self._cached_root = None

    # ------------------------------------------------------------------
    # Override: copy of parent choose_move with visit-count recording
    # ------------------------------------------------------------------
    def choose_move(self, state):
        n = self.n

        # ── Forced move check ──────────────────────────────────────────
        forced = check_forced_move(state)
        if forced is not None:
            state.nodes_visited = 1
            state.max_depth_reached = 0
            state.prunings = 0
            state.compute_ebf()
            self._cached_root = None
            # Peaked policy: 100% probability on the forced move
            policy_target = torch.zeros(n * n, dtype=torch.float32)
            r, c = forced
            policy_target[r * n + c] = 1.0
            state_enc = encode_board_flat(state.board, state.current_player, n).cpu()
            self._episode_buffer.append({
                'state_enc': state_enc,
                'policy_target': policy_target,
                'player': state.current_player,
            })
            return forced

        # ── Tree reuse ─────────────────────────────────────────────────
        root = self._find_reusable_root(state.last_move)
        if root is None:
            root_state = state.copy()
            root_state.result = Board.is_terminal(
                root_state.board, root_state.n, root_state.k, root_state.last_move
            )
            root = PUCTNode(root_state)
            self._expand(root)

        # ── Dirichlet noise at root ────────────────────────────────────
        if root.children:
            priors = torch.tensor([ch.prior for ch in root.children], dtype=torch.float32)
            noisy = self._add_dirichlet_noise(priors)
            for child, new_prior in zip(root.children, noisy.tolist()):
                child.prior = float(new_prior)

        # ── MCTS simulation loop ───────────────────────────────────────
        budget = SearchBudget(self.match_config, time.perf_counter_ns())
        max_depth = 0
        sim = 0
        while sim < self.num_simulations and not budget.exhausted(sim, 0):
            node = root
            depth = 0
            while node.is_expanded() and not node.is_terminal():
                node = max(node.children, key=lambda ch: ch.puct_score(self.c_puct))
                depth += 1
            max_depth = max(max_depth, depth)
            if not node.is_terminal():
                self._expand(node)
                x = encode_board_flat(node.state.board, node.state.current_player, self.n)
                with torch.no_grad():
                    _, value = self._net.forward(x)
            else:
                value = self._terminal_value(node.state)
            self._backpropagate(node, float(value))
            sim += 1

        # ── Temperature-annealed move selection ───────────────────────
        if not root.children:
            candidates = Board.get_candidate_moves(state, radius=2)
            move = candidates[0] if candidates else (0, 0)
            self._cached_root = None
        else:
            temp = self._effective_temperature(state.move_number)
            if temp < 1e-3:
                best_child = max(root.children, key=lambda ch: ch.visits)
            else:
                visits = torch.tensor([ch.visits for ch in root.children], dtype=torch.float32)
                visits_t = visits ** (1.0 / temp)
                total = visits_t.sum()
                probs = visits_t / total if total > 0 else visits_t
                idx = int(torch.multinomial(probs, 1).item())
                best_child = root.children[idx]
            move = best_child.move
            best_child.parent = None
            self._cached_root = best_child

        # ── Record policy target from root visit distribution ──────────
        visits_vec = torch.zeros(n * n, dtype=torch.float32)
        for ch in root.children:
            r, c = ch.move
            visits_vec[r * n + c] = float(ch.visits)
        total_v = visits_vec.sum()
        policy_target = visits_vec / total_v if total_v > 0 else visits_vec
        state_enc = encode_board_flat(state.board, state.current_player, n).cpu()
        self._episode_buffer.append({
            'state_enc': state_enc,
            'policy_target': policy_target,
            'player': state.current_player,
        })

        # ── Instrumentation (same as parent) ──────────────────────────
        state.nodes_visited = sim
        state.max_depth_reached = max_depth
        state.prunings = 0
        state.compute_ebf()
        return move

    def collect_examples(self, result: Result) -> list:
        """Convert episode buffer to labelled training examples.

        Args:
            result: Final game result (X_WINS / O_WINS / DRAW).

        Returns:
            List of (state_enc, policy_target, target_value) tuples.
            target_value ∈ {-1, 0, +1} from the perspective of the player
            who was to move at each position.
        """
        examples = []
        for entry in self._episode_buffer:
            player = entry['player']
            if result == Result.X_WINS:
                tv = 1.0 if player == Player.X else -1.0
            elif result == Result.O_WINS:
                tv = 1.0 if player == Player.O else -1.0
            else:  # DRAW
                tv = 0.0
            examples.append((entry['state_enc'], entry['policy_target'], tv))
        return examples


print('TrainableAlphaZeroAgent defined.')

In [ ]:
# ── Cell 7: Symmetry augmentation ────────────────────────────────────────
# Applies all 8 elements of the dihedral group D4 (4 rotations × 2 reflections)
# to each training example, giving an 8× dataset expansion for free.
#
# Works directly on the encoded state tensor (3, n, n) which already uses
# the own/opponent/ones channel convention expected by the network.
# This avoids the encoding mismatch in AlphaZeroAgent._generate_symmetries()
# which uses a different X=1/O=2 raw encoding.

def augment_examples(examples: list, n: int) -> list:
    """Augment training examples with all 8 D4 board symmetries.

    Args:
        examples: List of (state_enc: Tensor(3n^2), policy: Tensor(n^2), value: float).
        n: Board dimension.

    Returns:
        New list with 8x as many entries.
    """
    augmented = []
    for state_enc, policy_target, target_value in examples:
        # Reshape to spatial format
        spatial = state_enc.reshape(3, n, n)      # (3, n, n)
        policy_2d = policy_target.reshape(n, n)   # (n, n)

        for rot in range(4):
            # Rotation
            s_rot = torch.rot90(spatial, rot, dims=(1, 2))
            p_rot = torch.rot90(policy_2d, rot, dims=(0, 1))
            augmented.append((s_rot.flatten().clone(), p_rot.flatten().clone(), target_value))

            # Horizontal flip of the rotated version
            s_flip = torch.flip(s_rot, dims=[2])
            p_flip = torch.flip(p_rot, dims=[1])
            augmented.append((s_flip.flatten().clone(), p_flip.flatten().clone(), target_value))

    return augmented


print('augment_examples defined.')

In [ ]:
# ── Cell 8: Evaluation helpers ────────────────────────────────────────────

def evaluate_vs(agent, opponent, n_games: int, n: int, k: int) -> dict:
    """Evaluate agent against opponent, playing n_games/2 as X and n_games/2 as O.

    Returns dict with win_rate, draw_rate, loss_rate, wins, draws, losses, total.
    """
    wins = draws = losses = 0
    trainer = SelfPlayTrainer(n=n, k=k)
    half = n_games // 2

    # Play as X
    for _ in range(half):
        _, result = trainer.play_episode(agent, opponent)
        if result == Result.DRAW:
            draws += 1
        elif result == Result.X_WINS:
            wins += 1
        else:
            losses += 1

    # Play as O
    for _ in range(half):
        _, result = trainer.play_episode(opponent, agent)
        if result == Result.DRAW:
            draws += 1
        elif result == Result.O_WINS:
            wins += 1
        else:
            losses += 1

    total = wins + draws + losses
    return {
        'win_rate': wins / total,
        'draw_rate': draws / total,
        'loss_rate': losses / total,
        'wins': wins, 'draws': draws, 'losses': losses, 'total': total,
    }


def compute_policy_entropy(agent, n_samples: int, n: int, k: int) -> float:
    """Sample random non-terminal positions and compute average policy entropy.

    Returns the mean entropy in nats. Maximum possible: log(n*n).
    Lower entropy = more decisive/confident network.
    """
    trainer = SelfPlayTrainer(n=n, k=k)
    entropies = []
    rand_agent = RandomAgent()

    for _ in range(n_samples):
        # Play a random partial game to get a mid-game position
        board = Board.create(n)
        from tictactoe.core.state import GameState
        state = GameState(board=board, current_player=Player.X, n=n, k=k)
        num_moves = random.randint(0, n * n // 2)
        for _ in range(num_moves):
            if state.result != Result.IN_PROGRESS:
                break
            candidates = Board.get_candidate_moves(state, radius=2)
            if not candidates:
                break
            state = state.apply_move(random.choice(candidates))
            state.result = Board.is_terminal(state.board, n, k, state.last_move)
        if state.result != Result.IN_PROGRESS:
            continue

        x = encode_board_flat(state.board, state.current_player, n)
        with torch.no_grad():
            logits, _ = agent._net.forward(x)
            pi = torch.softmax(logits, dim=-1).cpu()
        entropy = float(-(pi * (pi + 1e-8).log()).sum())
        entropies.append(entropy)

    return sum(entropies) / len(entropies) if entropies else 0.0


print('Evaluation helpers defined.')

In [ ]:
# ── Cell 9: Checkpoint helpers ────────────────────────────────────────────

def save_checkpoint(agent, history: dict, iteration: int, drive_dir: str):
    """Save model weights (.pt) and training history (_meta.json) to Drive."""
    ckpt_path = os.path.join(drive_dir, f'checkpoint_iter_{iteration:04d}')
    agent.save(ckpt_path)  # saves as ckpt_path + '.pt'
    meta = {
        'iteration': iteration,
        'n': N, 'k': K,
        'network_type': NETWORK_TYPE,
        'num_simulations': NUM_SIMULATIONS,
        'history': history,
        'timestamp': time.strftime('%Y-%m-%dT%H:%M:%S'),
    }
    meta_path = ckpt_path + '_meta.json'
    with open(meta_path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  Saved: {ckpt_path}.pt + _meta.json')


def load_checkpoint(agent, path: str) -> tuple:
    """Load weights and history from a checkpoint (path without .pt extension).

    Returns (history dict, start_iteration int).
    """
    agent.load(path)  # loads path + '.pt'
    meta_path = path + '_meta.json'
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f'  Resumed from iteration {meta["iteration"]}')
        return meta.get('history', {}), meta['iteration']
    print('  No metadata file found — starting fresh history.')
    return {}, 0


print('Checkpoint helpers defined.')

In [ ]:
# ── Cell 10: Instantiate agent and optional resume ────────────────────────

agent = TrainableAlphaZeroAgent(
    n=N, k=K,
    num_simulations=NUM_SIMULATIONS,
    c_puct=C_PUCT,
    temperature=TEMPERATURE,
    lr=LR,
    network_type=NETWORK_TYPE,
)
print(f'Agent: {agent.get_name()}')
n_params = sum(p.numel() for p in agent._net.parameters())
print(f'Network parameters: {n_params:,}')
print(f'Network device: {next(agent._net.parameters()).device}')

# Metrics history — each list grows by 1 per iteration
_empty_history = {
    'iteration': [],
    'avg_loss': [],
    'vs_random_wr': [],
    'vs_minimax_wr': [],
    'draw_rate_selfplay': [],
    'avg_game_length': [],
    'replay_buffer_size': [],
    'policy_entropy': [],
    'iteration_time_s': [],
}
history = {k: list(v) for k, v in _empty_history.items()}  # deep copy
start_iteration = 0
replay_buffer: list = []

if RESUME_FROM is not None:
    loaded_history, start_iteration = load_checkpoint(agent, RESUME_FROM)
    if loaded_history:
        # Merge loaded keys; preserve any keys not yet in history
        for k in history:
            if k in loaded_history:
                history[k] = loaded_history[k]
    print(f'Resuming from iteration {start_iteration}.')
    print('Note: replay buffer starts empty — it will refill in the first iteration.')
else:
    print('Starting fresh training.')

In [ ]:
# ── Cell 11: Main training loop ───────────────────────────────────────────
# Each iteration:
#   1. SELF-PLAY: run GAMES_PER_ITERATION games, collect (state, policy, value)
#   2. AUGMENT: apply 8 D4 symmetries (8x expansion)
#   3. CAP: trim replay buffer to BUFFER_MAX_SIZE (rolling window)
#   4. TRAIN: shuffle examples, iterate mini-batches via train_on_batch
#   5. EVAL (every EVAL_EVERY): win rate vs Random and MinimaxAB
#   6. CHECKPOINT (every CHECKPOINT_EVERY): save to Google Drive

random_opponent = RandomAgent()
minimax_opponent = MinimaxAB(depth=4)  # near-perfect oracle at n=3
trainer = SelfPlayTrainer(n=N, k=K)

end_iteration = start_iteration + NUM_ITERATIONS

print(f'Training iterations: {start_iteration} → {end_iteration}')
print(f'{"Iter":>6} | {"Loss":>8} | {"vs Rand":>8} | {"vs Mini":>8} | {"DrawRate":>9} | {"Entropy":>8} | {"Time(s)":>7}')
print('-' * 73)

for iteration in range(start_iteration, end_iteration):
    iter_start = time.time()

    # ── 1. SELF-PLAY ──────────────────────────────────────────────────
    raw_examples = []
    game_lengths = []
    draws_this_iter = 0

    for _ in range(GAMES_PER_ITERATION):
        agent.clear_episode_buffer()
        move_history, result = trainer.play_episode(agent, agent)
        game_lengths.append(len(move_history))
        if result == Result.DRAW:
            draws_this_iter += 1
        raw_examples.extend(agent.collect_examples(result))

    # ── 2. AUGMENT (8x D4 symmetries) ────────────────────────────────
    augmented = augment_examples(raw_examples, N)
    replay_buffer.extend(augmented)

    # ── 3. CAP (rolling window) ───────────────────────────────────────
    if len(replay_buffer) > BUFFER_MAX_SIZE:
        replay_buffer = replay_buffer[-BUFFER_MAX_SIZE:]

    # ── 4. TRAIN ──────────────────────────────────────────────────────
    indices = list(range(len(replay_buffer)))
    random.shuffle(indices)
    total_loss = 0.0
    num_batches = 0
    for start in range(0, len(indices), BATCH_SIZE):
        batch_idx = indices[start:start + BATCH_SIZE]
        if len(batch_idx) < 2:  # skip degenerate last batch
            continue
        batch = [replay_buffer[i] for i in batch_idx]
        try:
            loss = agent.train_on_batch(batch, lr=LR)
            total_loss += loss
            num_batches += 1
        except Exception as e:
            print(f'  [warn] batch training error: {e}')
    avg_loss = total_loss / max(num_batches, 1)

    # ── 5. METRICS ────────────────────────────────────────────────────
    draw_rate_sp = draws_this_iter / GAMES_PER_ITERATION
    avg_game_len = sum(game_lengths) / len(game_lengths)
    iter_time = time.time() - iter_start

    history['iteration'].append(iteration)
    history['avg_loss'].append(avg_loss)
    history['draw_rate_selfplay'].append(draw_rate_sp)
    history['avg_game_length'].append(avg_game_len)
    history['replay_buffer_size'].append(len(replay_buffer))
    history['iteration_time_s'].append(iter_time)

    # ── 6. EVALUATION (every EVAL_EVERY) ─────────────────────────────
    do_eval = ((iteration + 1) % EVAL_EVERY == 0)
    if do_eval:
        r_eval = evaluate_vs(agent, random_opponent, EVAL_GAMES, N, K)
        m_eval = evaluate_vs(agent, minimax_opponent, EVAL_GAMES, N, K)
        entropy = compute_policy_entropy(agent, n_samples=50, n=N, k=K)
        history['vs_random_wr'].append(r_eval['win_rate'])
        history['vs_minimax_wr'].append(m_eval['win_rate'])
        history['policy_entropy'].append(entropy)
        print(f'{iteration+1:>6} | {avg_loss:>8.4f} | {r_eval["win_rate"]:>8.3f} | '
              f'{m_eval["win_rate"]:>8.3f} | {draw_rate_sp:>9.3f} | '
              f'{entropy:>8.4f} | {iter_time:>7.1f}')
    else:
        history['vs_random_wr'].append(None)
        history['vs_minimax_wr'].append(None)
        history['policy_entropy'].append(None)
        if (iteration + 1) % 10 == 0:
            print(f'{iteration+1:>6} | {avg_loss:>8.4f} | {"--":>8} | {"--":>8} | '
                  f'{draw_rate_sp:>9.3f} | {"--":>8} | {iter_time:>7.1f}')

    # ── 7. CHECKPOINT (every CHECKPOINT_EVERY) ────────────────────────
    if (iteration + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(agent, history, iteration + 1, DRIVE_CHECKPOINT_DIR)

# Final checkpoint
print('\nTraining complete!')
save_checkpoint(agent, history, end_iteration, DRIVE_CHECKPOINT_DIR)

In [ ]:
# ── Cell 12: Training curves ──────────────────────────────────────────────

def plot_training_curves(history: dict, n: int, k: int, save_dir: str):
    iters = history['iteration']
    eval_iters = [i for i, v in zip(iters, history['vs_random_wr']) if v is not None]
    wr_random = [v for v in history['vs_random_wr'] if v is not None]
    wr_minimax = [v for v in history['vs_minimax_wr'] if v is not None]
    entropy_iters = [i for i, v in zip(iters, history['policy_entropy']) if v is not None]
    entropy_vals = [v for v in history['policy_entropy'] if v is not None]

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f'AlphaZero Training — n={n}, k={k}, sims={NUM_SIMULATIONS}', fontsize=14, fontweight='bold')

    # Loss
    ax = axes[0, 0]
    ax.plot(iters, history['avg_loss'], 'b-', linewidth=1)
    ax.set_title('Training Loss (Policy CE + Value MSE)')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Loss'); ax.grid(True, alpha=0.4)

    # Win rates
    ax = axes[0, 1]
    if eval_iters:
        ax.plot(eval_iters, wr_random, 'g-o', markersize=4, label='vs Random')
        ax.plot(eval_iters, wr_minimax, 'r-o', markersize=4, label='vs MinimaxAB')
        ax.axhline(0.95, color='green', linestyle='--', alpha=0.4, label='95% target (vs Random)')
        ax.axhline(0.50, color='red', linestyle='--', alpha=0.4, label='50% target (vs Minimax)')
    ax.set_title('Win Rate vs Opponents')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Win Rate')
    ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8); ax.grid(True, alpha=0.4)

    # Self-play draw rate
    ax = axes[0, 2]
    ax.plot(iters, history['draw_rate_selfplay'], color='purple', linewidth=1)
    ax.axhline(0.70, color='gray', linestyle='--', alpha=0.5, label='70% target')
    ax.set_title('Self-Play Draw Rate')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Draw Rate')
    ax.set_ylim(-0.05, 1.05); ax.legend(fontsize=8); ax.grid(True, alpha=0.4)

    # Average game length
    ax = axes[1, 0]
    ax.plot(iters, history['avg_game_length'], color='orange', linewidth=1)
    ax.set_title('Average Game Length (moves)')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Moves'); ax.grid(True, alpha=0.4)

    # Policy entropy
    ax = axes[1, 1]
    if entropy_iters:
        ax.plot(entropy_iters, entropy_vals, color='teal', marker='o', markersize=4)
        max_entropy = math.log(n * n)
        ax.axhline(max_entropy, color='gray', linestyle='--', alpha=0.4, label=f'Uniform ({max_entropy:.2f} nats)')
    ax.set_title('Policy Entropy (lower = more decisive)')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Entropy (nats)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.4)

    # Iteration time
    ax = axes[1, 2]
    ax.plot(iters, history['iteration_time_s'], color='brown', linewidth=1)
    ax.set_title('Time per Iteration (seconds)')
    ax.set_xlabel('Iteration'); ax.set_ylabel('Seconds'); ax.grid(True, alpha=0.4)

    plt.tight_layout()
    png_path = os.path.join(save_dir, 'training_curves.png')
    plt.savefig(png_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved to {png_path}')


plot_training_curves(history, N, K, DRIVE_CHECKPOINT_DIR)

In [ ]:
# ── Cell 13: Final policy entropy diagnostic ──────────────────────────────
# Sample 200 random positions and measure average entropy to assess
# how decisive the trained network is compared to a uniform distribution.

entropy = compute_policy_entropy(agent, n_samples=200, n=N, k=K)
max_entropy = math.log(N * N)

print(f'Average policy entropy: {entropy:.4f} nats')
print(f'Uniform maximum:        {max_entropy:.4f} nats')
print(f'Relative entropy:       {entropy / max_entropy:.1%} of maximum')
print()
if entropy / max_entropy < 0.5:
    print('Good: network has learned to concentrate probability on strong moves.')
elif entropy / max_entropy < 0.8:
    print('Moderate: network shows some preference but still fairly uniform — consider more training.')
else:
    print('High entropy: network is still close to uniform — more training needed.')